# **Europe PMC API Data Acquisition Pipeline**
---
**Author** :: **James Chukwuemeka**

**Affiliation** :: [University of Salford UK](https://www.salford.ac.uk/)

**Connect on GitHub** :: [GitHub](https://github.com/Chukwuemeka-James)

**Connect on Linkedin** :: [LinkedIn](https://www.linkedin.com/in/chukwuemekajames/)

---

**Fetching and Processing Research Abstracts from Europe PMC**

This notebook demonstrates how to interact with the official Europe PMC RESTful API to fetch, clean, and process research paper metadata and abstracts based on a targeted search query.

**Key Features:**

1. **Retrieve Research Records:** Sends HTTP GET requests to the Europe PMC API, fetches JSON data, and extracts relevant information such as abstracts, publication years, affiliations, and MeSH headings.  
2. **Balanced Dataset Collection:** Iterates through specified years to collect a uniform number of records per year, ensuring the final dataset is not skewed toward recent publication spikes.  
3. **Resilient Data Extraction:** Implements an exponential backoff strategy to gracefully handle server errors and API rate limits.  
4. **Data Flattening:** Safely parses complex, deeply nested JSON arrays into clean, semicolon-separated strings.  
5. **Modular and Efficient:** Implements a robust `EuropePMCClient` class for fetching, parsing, and saving records, ensuring reusability and efficient batch processing.

**How It Works:**

- Define a search query, for example, targeting "United Kingdom" affiliations and health-related keywords.
- Fetch and paginate through the API using `cursorMark` to bypass standard pagination limits.
- Extract the nested metadata and abstract content for each paper.
- Save the processed data in a structured CSV format.
- Use the extracted data for further analysis or natural language processing tasks.

This workflow automates research paper retrieval and processing, making it a highly valuable tool for analysing and understanding health reseaerch trend.

In [ ]:
# Install depencies

!pip install requests
!pip install tqdm

In [ ]:
# Imports

import time
import logging
import csv
import requests
from pathlib import Path
from typing import List, Dict, Any, Optional
from requests.exceptions import RequestException
from tqdm import tqdm

# Configure basic logging for visibility during long API pulls
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

### Define the **EuropePMCClient**
This will be used as a data extraction pipeline client for the Europe PMC RESTful API.


To help you understand the data points your `EuropePMCClient` is gathering, here is a concise breakdown of each feature:

* **id**: The unique identifier (such as a PMID or PMCID) used to specifically reference and track each individual research record within the global database.
* **pubYear**: The calendar year the article was officially published, which allows you to verify that your dataset is perfectly balanced across your 2017-2026 timeline.
* **abstractText**: A condensed summary of the study's background, methods, and findings, serving as the core text for any future analysis or machine learning tasks.
* **authorAffiliationsList**: The professional institutions or universities associated with the authors, confirming that the research is tied to the "United Kingdom" as per your query.
* **meshHeadingList**: Standardized "Medical Subject Headings" (MeSH) used by librarians to categorize the paper under specific, controlled clinical and biological topics.
* **keywordList**: Unstructured, natural language terms provided directly by the authors to highlight the most important themes and subjects of their work.



These features combined provide a comprehensive detail of every research.

In [ ]:
class EuropePMCClient:
    """
    A data extraction pipeline client for the Europe PMC RESTful API.
    """

    BASE_URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    def __init__(self, max_retries: int = 5):
        """
        Initializes the API client with retry logic and directory setup.

        Parameters:
            max_retries (int): The maximum number of retry attempts for failed API calls.
        """
        self.max_retries = max_retries
        self.output_dir = Path("research_data")

        # Ensure the output directory exists
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.output_file = self.output_dir / "uk_health_research_balanced.csv"

    def fetch_page(self, query: str, cursor_mark: str) -> Optional[Dict[str, Any]]:
        """
        Fetches a single page of results from the API with exponential backoff.

        Parameters:
            query (str): The search query string formatted for Europe PMC.
            cursor_mark (str): The pagination cursor for deep paging.

        Returns:
            dict or None: A dictionary containing the JSON response from the API,
                          or None if the request ultimately fails.
        """
        params = {
            "query": query,
            "format": "json",
            "resultType": "core",
            "cursorMark": cursor_mark,
            "pageSize": 1000  # Max allowed by Europe PMC
        }

        for attempt in range(self.max_retries):
            try:
                response = requests.get(self.BASE_URL, params=params, timeout=15)

                # Handle rate limiting (429) or server errors (5xx)
                if response.status_code in [429, 500, 502, 503, 504]:
                    wait_time = 2 ** attempt
                    logging.warning(f"HTTP {response.status_code} received. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                    continue

                response.raise_for_status()
                return response.json()

            except RequestException as e:
                logging.error(f"Request failed (Attempt {attempt + 1}/{self.max_retries}): {e}")
                time.sleep(2 ** attempt)

        logging.error("Max retries exceeded. Halting pagination.")
        return None

    @staticmethod
    def _safe_extract_list(data: Dict[str, Any], list_key: str, item_key: str, value_key: str) -> str:
        """
        Helper to extract nested list values into a semicolon-separated string.

        Parameters:
            data (dict): The dictionary containing the nested list.
            list_key (str): The top-level key containing the list.
            item_key (str): The inner key representing individual items.
            value_key (str): The specific target value to extract.

        Returns:
            str: A semicolon-separated string of the extracted values.
        """
        try:
            items = data.get(list_key, {}).get(item_key, [])
            if isinstance(items, dict):
                items = [items]

            extracted = [item.get(value_key, "") for item in items if isinstance(item, dict) and value_key in item]
            return "; ".join(filter(None, extracted))
        except Exception:
            return ""

    def parse_article(self, article: Dict[str, Any]) -> Dict[str, str]:
        """
        Extracts and flattens specific fields from a complex article JSON object.

        Parameters:
            article (dict): A dictionary representing a single article's metadata.

        Returns:
            dict: A flattened dictionary containing targeted data fields.
        """
        affiliations = []
        authors = article.get("authorList", {}).get("author", [])
        if isinstance(authors, dict):
            authors = [authors]

        for author in authors:
            aff_details = author.get("authorAffiliationDetailsList", {}).get("authorAffiliation", [])
            if isinstance(aff_details, dict):
                aff_details = [aff_details]
            for aff in aff_details:
                if "affiliation" in aff:
                    affiliations.append(aff["affiliation"])

        return {
            "id": article.get("id", ""),
            "abstractText": article.get("abstractText", "").strip(),
            "authorAffiliationsList": "; ".join(set(affiliations)),
            "pubYear": article.get("pubYear", ""),
            "meshHeadingList": self._safe_extract_list(article, "meshHeadingList", "meshHeading", "descriptorName"),
            "keywordList": self._safe_extract_list(article, "keywordList", "keyword", "keyword")
        }

    def collect_balanced_dataset(self, base_query: str, years: List[int], records_per_year: int) -> List[Dict[str, str]]:
        """
        Orchestrates fetching a specific number of records per year to ensure a balanced dataset.

        Parameters:
            base_query (str): The foundational search query string.
            years (list): A list of integers representing the years to sample.
            records_per_year (int): The target number of articles to extract per year.

        Returns:
            list: A list of dictionaries, where each dictionary is a parsed article.
        """
        all_clean_data = []
        total_target = len(years) * records_per_year

        logging.info(f"Starting balanced collection: {records_per_year} records/year for {len(years)} years ({total_target} total).")

        with tqdm(total=total_target, desc="Collecting Records") as pbar:
            for year in years:
                # Dynamically append the publication year to the query
                year_query = f'({base_query}) AND PUB_YEAR:{year}'
                cursor_mark = "*"
                year_data = []

                while len(year_data) < records_per_year:
                    data = self.fetch_page(year_query, cursor_mark)

                    if not data or "resultList" not in data or not data["resultList"]["result"]:
                        logging.warning(f"Exhausted results for {year}. Only found {len(year_data)} abstracts.")
                        break

                    results = data["resultList"]["result"]

                    for article in results:
                        parsed_record = self.parse_article(article)

                        # Only keep records with an actual abstract
                        if parsed_record["abstractText"]:
                            year_data.append(parsed_record)
                            all_clean_data.append(parsed_record)
                            pbar.update(1)

                            if len(year_data) == records_per_year:
                                break

                    # If we hit our yearly target in the middle of a page, break the while loop
                    if len(year_data) == records_per_year:
                        break

                    # Update cursor for the next page
                    next_cursor = data.get("nextCursorMark")
                    if not next_cursor or next_cursor == cursor_mark:
                        logging.info(f"Reached end of available cursor marks for {year}.")
                        break
                    cursor_mark = next_cursor

                    # Polite delay between requests
                    time.sleep(0.5)

        logging.info(f"Successfully collected {len(all_clean_data)} balanced records.")
        return all_clean_data

    def save_to_csv(self, data: List[Dict[str, str]]) -> None:
        """
        Saves the structured dictionary data to a CSV file.

        Parameters:
            data (list): A list of dictionaries representing the extracted records.
        """
        if not data:
            logging.warning("No data to save.")
            return

        fieldnames = ["id", "pubYear", "abstractText", "authorAffiliationsList", "meshHeadingList", "keywordList"]

        try:
            with open(self.output_file, mode="w", encoding="utf-8", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(data)
            logging.info(f"Dataset securely saved to: {self.output_file.absolute()}")
        except IOError as e:
            logging.error(f"Failed to write to CSV: {e}")

### Run the code

In [ ]:
def main():
    """
    Main function to orchestrate the Europe PMC data acquisition pipeline.

    Workflow:
    1. Defines the base search query targeting UK affiliations and health/medical keywords.
    2. Establishes the temporal scope (e.g., 2017 to 2026) and the quota per year.
    3. Instantiates the EuropePMCClient.
    4. Collects the balanced dataset dynamically.
    5. Outputs the final, cleaned data to a structured CSV file.
    """
    # Define the base query targeting UK affiliations and health/medical keywords
    BASE_QUERY = 'AFF:"United Kingdom" AND (health OR medical OR clinical)'

    # 2017 to 2026 = 10 years. 10 years * 500 records = 5,000 total records.
    # Adjust YEARS and RECORDS_PER_YEAR based on required dataset size
    YEARS = list(range(2017, 2027))
    RECORDS_PER_YEAR = 500

    print(f"Initializing EuropePMC Client...")
    print(f"Targeting {RECORDS_PER_YEAR} records per year between {YEARS[0]} and {YEARS[-1]}")

    # Initialize and run the pipeline
    client = EuropePMCClient()
    dataset = client.collect_balanced_dataset(
        base_query=BASE_QUERY,
        years=YEARS,
        records_per_year=RECORDS_PER_YEAR
    )

    print("\nWriting Data to CSV...")
    # Save the output
    client.save_to_csv(dataset)
    print("Process completed!")

if __name__ == "__main__":
    main()


Initializing EuropePMC Client...
Targeting 500 records per year between 2017 and 2026



Writing Data to CSV...
Process completed!


### View the Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("research_data/uk_health_research_balanced.csv")
df.head(10)

,id,pubYear,abstractText,authorAffiliationsList,meshHeadingList,keywordList
0,29113630,2017,IntroductionThe United Kingdom is in the fourt...,"University of Strathclyde, Glasgow, United Kin...",Humans; Influenza A virus; Influenza B virus; ...,NaN
1,29240772,2017,<h4>Introduction</h4>It has been suggested tha...,"Portsmouth Hospitals NHS Trust, Portsmouth, Un...","Humans; Death, Sudden, Cardiac; Monitoring, Ph...",NaN
2,29261655,2017,<h4>Background</h4>Excessive haemorrhage at ce...,"Royal Victoria Infirmary, Newcastle-upon-Tyne,...","Humans; Blood Loss, Surgical; Prognosis; Treat...",NaN
3,28838399,2017,<h4>Objectives</h4>Despite advances in novel d...,"Royal Marsden NHS Foundation Trust, London, Un...","Humans; Carcinoma, Non-Small-Cell Lung; Lung N...",NaN
4,29088308,2017,Motivational interviewing (MI) is a method for...,"Wessex NIHR CLAHRC, Southampton, United Kingdo...",Humans; Respiratory Therapy; Motivation; Quali...,NaN
5,29252995,2017,<h4>Objectives</h4>New point of care diagnosti...,"Tropical and Infectious Disease Unit, Royal Li...",Humans; Bacterial Infections; Respiratory Trac...,NaN
6,29267304,2017,<h4>Objective</h4>To estimate the prevalence o...,University Hospitals Leicester NHS Foundation ...,"Humans; Otitis Media, Suppurative; Hearing Los...",NaN
7,29023584,2017,<h4>Introduction</h4>Recent National Institute...,"Neonatal Intensive Care Unit, Royal Jubilee Ma...",Humans; Hyponatremia; Fluid Therapy; Medicatio...,NaN
8,29029812,2017,"The United Kingdom, like all European countrie...","London School of Paediatrics, United Kingdom.;...",Humans; Child Mortality; Government; Governmen...,NaN
9,28671982,2017,<h4>Background</h4>Intellectual disability (ID...,Cambridgeshire and Peterborough National Healt...,Humans; Epilepsy; Cohort Studies; Quality of L...,NaN


In [ ]:
# Check data completeness based on year column

df["pubYear"].value_counts()

,count
pubYear,
2017,500
2018,500
2019,500
2020,500
2021,500
2022,500
2023,500
2024,500
2025,500
